# Phase 2 — Trace the count signal (layer x timestep)
Phase 1 showed the TEXT drives the count but realizes it unreliably (over-generation). Here we find WHERE and WHEN the count is committed, and whether the break is **image-side** (self-attn `attn1`), the **matching** junction (cross-attn `attn2`), or **text**.

For each image we capture pooled activations at curated sites at a few timesteps, then measure how decodable the **requested** vs **rendered** count is at each (site, step).

**Runtime:** GPU.

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os
from src.prompts import generate_grid
from src.pipeline import (load_sdxl, catalog_attention_sites,
                          select_probe_sites, generate_and_capture)
from src.detector import Detector
from src.scoring import count_from_detections
from src.probes import decodability_map, fit_eval_magnitude
from src.config import load_config

In [ ]:
cfg = load_config('configs/phase2.yaml')
grid = generate_grid(cfg.counts, cfg.objects, cfg.seeds)
print(len(grid), 'images; capture steps:', cfg.capture_steps)

In [ ]:
pipe = load_sdxl()
det = Detector()
sites = select_probe_sites(catalog_attention_sites(pipe.unet))
print(len(sites), 'probe sites (attn1=image-side, attn2=matching):')
print(sites)

In [ ]:
# Generate each image, capture pooled activations at the chosen steps,
# and score the rendered count with the detector.
rows = []
feats = {st: {} for st in cfg.capture_steps}
for i, p in enumerate(grid):
    img, snaps = generate_and_capture(pipe, p.text, p.seed, sites,
                                      cfg.capture_steps, cfg.num_inference_steps)
    rendered = count_from_detections(det.detect(img, [p.obj]), p.obj,
                                     cfg.score_threshold)
    rows.append({'obj': p.obj, 'count': p.count, 'seed': p.seed,
                 'rendered': rendered})
    for st in cfg.capture_steps:
        for s, v in snaps.get(st, {}).items():
            feats[st].setdefault(s, []).append(v)
    if (i + 1) % 20 == 0:
        print(f'{i+1}/{len(grid)}')
df = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
df.to_csv('results/phase2_counts.csv', index=False)
df.head()

In [ ]:
# Probe requested vs rendered count decodability at every (step, site).
y_req = df['count'].to_numpy(float)
y_ren = df['rendered'].to_numpy(float)
req_map, ren_map = {}, {}
for st in cfg.capture_steps:
    X = {s: np.array(feats[st][s]) for s in sites if s in feats[st]}
    req_map[st] = decodability_map(X, y_req, mode='magnitude')
    ren_map[st] = decodability_map(X, y_ren, mode='magnitude')
print('done probing', len(sites), 'sites x', len(cfg.capture_steps), 'steps')

In [ ]:
# (H3) Is the requested count in the fused TEXT embedding? (defensive)
try:
    tf = []
    for p in grid:
        pe = pipe.encode_prompt(prompt=p.text, prompt_2=None,
                                device=pipe.device, num_images_per_prompt=1,
                                do_classifier_free_guidance=False)
        tf.append(pe[0][0].mean(0).detach().cpu().float().numpy())
    r2 = fit_eval_magnitude(np.array(tf), y_req)['r2']
    print(f'TEXT embedding -> requested count: R2 = {r2:.2f}')
except Exception as e:
    print('text-embedding probe skipped:', repr(e))

In [ ]:
# Heatmaps: requested R2, rendered R2, and the gap (req - ren).
import matplotlib.pyplot as plt
def short(s):
    return (s.replace('down_blocks','down').replace('up_blocks','up')
             .replace('mid_block','mid').replace('.attentions','.a')
             .replace('.transformer_blocks.0',''))
labels = [short(s) for s in sites]
steps = cfg.capture_steps
mat_req = np.array([[req_map[st].get(s, np.nan) for st in steps] for s in sites])
mat_ren = np.array([[ren_map[st].get(s, np.nan) for st in steps] for s in sites])
gap = mat_req - mat_ren
fig, axes = plt.subplots(1, 3, figsize=(13, max(4, len(sites) * 0.34)))
panels = [(mat_req, 'requested count R2', 'viridis', 0, 1),
          (mat_ren, 'rendered count R2', 'viridis', 0, 1),
          (gap, 'gap = req - ren', 'coolwarm', -1, 1)]
for ax, (mat, title, cmap, vmin, vmax) in zip(axes, panels):
    im = ax.imshow(mat, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(steps))); ax.set_xticklabels([f's{st}' for st in steps])
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=6)
    ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('results/phase2_trace.png', dpi=110, bbox_inches='tight'); plt.show()

In [ ]:
# Where does the RENDERED (eventual) count first become decodable?
best = max(((ren_map[st][s], s, st) for st in steps for s in sites
            if s in ren_map[st]), default=(0, None, None))
print('peak rendered-count decodability: R2=%.2f at %s (step %s)'
      % (best[0], best[1], best[2]))
for st in steps:
    a1 = np.nanmean([ren_map[st][s] for s in sites if s.endswith('attn1') and s in ren_map[st]])
    a2 = np.nanmean([ren_map[st][s] for s in sites if s.endswith('attn2') and s in ren_map[st]])
    print(f'step {st}: mean rendered-R2  attn1(image)={a1:.2f}  attn2(matching)={a2:.2f}')

## How to read this
- **Requested-count R2 high on `attn2` but low on `attn1`** = the number is present only in the text/matching channel, not yet built image-side (the text-leak we must control for).
- **Rendered-count R2 low early, rising at a specific (site, step)** = the **commitment / realization point** — where the eventual (often wrong) count gets written into the image. That site+step is the Phase-4 intervention target.
- **Large positive gap (req >> ren)** = the model 'knows' the number there but hasn't realized it yet -> the break is downstream of that point.
- **attn1 vs attn2 means (last cell):** if rendered count is carried mainly by `attn2` and only late, the failure lives in **matching/realization** (Phase-1-consistent); if `attn1` carries it early, the image commits early and the fix belongs upstream.